In [ ]:
import pandas as pd
import numpy as np
from prefixspan import PrefixSpan
from pymining import seqmining
import math

In [71]:
df = pd.read_csv("C:/Users/Kishan Pawar/Downloads/transactions.csv",encoding='latin1')
df

,ï»¿user_id,sequence
0,1,"A,B,C,D"
1,2,"A,B,E"
2,3,"A,B,C"
3,4,"B,C,E"


In [72]:
df.isnull().sum()

ï»¿user_id    0
sequence      0
dtype: int64

# SPADE

In [73]:
min_support = 40/100 
min_support = math.ceil(min_support * df.shape[1])
df["sequence"] = df["sequence"].str.replace(",", "", regex=False)
# Running SPADE-like frequent sequence mining
freq_seqs = seqmining.freq_seq_enum(df["sequence"], min_support)


In [74]:
print("Frequent Sequential Patterns (SPADE):")
for seq, freq in freq_seqs:
    print(seq, "-> support:", freq)


Frequent Sequential Patterns (SPADE):
('B', 'C', 'D') -> support: 1
('A',) -> support: 3
('D',) -> support: 1
('A', 'D') -> support: 1
('B', 'C', 'E') -> support: 1
('E',) -> support: 2
('B', 'D') -> support: 1
('C', 'D') -> support: 1
('A', 'B') -> support: 3
('B', 'C') -> support: 3
('A', 'C', 'D') -> support: 1
('A', 'B', 'C') -> support: 2
('A', 'B', 'E') -> support: 1
('B', 'E') -> support: 2
('C', 'E') -> support: 1
('A', 'B', 'C', 'D') -> support: 1
('C',) -> support: 3
('B',) -> support: 4
('A', 'B', 'D') -> support: 1
('A', 'C') -> support: 2
('A', 'E') -> support: 1


# GSP

In [75]:
from itertools import combinations

In [76]:
def is_subsequence(subseq, seq):
    it = iter(seq) 
    return all(c in it for c in subseq)

In [77]:
def gsp(sequences, min_support):
    # Step 1: Generate 1-sequences
    items = set(item for seq in sequences for item in seq)
    freq_patterns = [{(i,): sum(is_subsequence((i,), seq) for seq in sequences)} for i in items]
    freq_patterns = [fp for fp in freq_patterns if list(fp.values())[0] >= min_support]

    results = []
    k = 1
    while freq_patterns:
        results.extend(freq_patterns)

        # Step 2: Generate candidate (k+1)-sequences
        candidates = []
        freq_items = [list(fp.keys())[0] for fp in freq_patterns]
        for a, b in combinations(freq_items, 2):
            if a[:-1] == b[:-1]:
               candidates.append(a + (b[-1],))
               freq_patterns = []
        for cand in candidates:
            support = sum(is_subsequence(cand, seq) for seq in sequences)
            if support >= min_support:
                freq_patterns.append({cand: support})

        k += 1

    return results

sequences = [
    ['a', 'b', 'c'],
    ['a', 'c'],
    ['b', 'c', 'a'],
    ['a', 'b', 'c', 'd']
]
min_support = 2
frequent_patterns = gsp(sequences, min_support)

print("Frequent Sequential Patterns (GSP):")
for pattern in frequent_patterns:
    for seq, support in pattern.items():
        print(seq, "-> support:", support)



Frequent Sequential Patterns (GSP):
('c',) -> support: 4
('b',) -> support: 3
('a',) -> support: 4


# Prefix

In [78]:
ps = PrefixSpan(df['sequence'])
ps.minlen = 1
frequent_patterns = ps.frequent(2)
frequent_patterns.sort(reverse=False)
for support, pattern in frequent_patterns[:25]:
    print(f"Support: {support}, Pattern: {pattern}")

Support: 2, Pattern: ['A', 'B', 'C']
Support: 2, Pattern: ['A', 'C']
Support: 2, Pattern: ['B', 'E']
Support: 2, Pattern: ['E']
Support: 3, Pattern: ['A']
Support: 3, Pattern: ['A', 'B']
Support: 3, Pattern: ['B', 'C']
Support: 3, Pattern: ['C']
Support: 4, Pattern: ['B']
